# 01 — Reproducible Raw-Data Audit

**Objective:** Audit the official raw OpenML data and quantify the memory and precision effects of an explicit `float64` to `float32` conversion.

No split, preprocessing, feature removal, visualization, or modeling is performed.

In [ ]:
from pathlib import Path
import sys

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs" / "config.yaml").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import (
    audit_numeric_features,
    compare_numeric_precision,
    convert_numeric_dtype,
    get_dataset_summary,
    load_dataset,
    validate_dtype_conversion,
)
from src.validation import create_train_test_split, validate_train_test_split

## Raw loading and source metadata

The shared loader is called once with optimization disabled, preserving OpenML values and dtypes. The project and OpenML dataset names remain separate.

In [ ]:
X_raw, y, metadata = load_dataset(optimize_memory=False)
summary = get_dataset_summary(X_raw, y, metadata)
{
    "project_dataset_name": metadata["project_dataset_name"],
    "openml_dataset_name": metadata["openml_dataset_name"],
    "openml_id": metadata["openml_id"],
    "target_name": metadata["target_name"],
    "dimensions": (summary["n_rows"], summary["n_features"]),
    "feature_names_preview": metadata["feature_names"][:8],
}

## Types and global data quality

In [ ]:
quality_fields = [
    "missing_values_X", "missing_values_y", "duplicate_rows_X",
    "duplicate_column_name_count", "constant_feature_count",
    "quasi_constant_feature_count", "quasi_constant_threshold",
    "infinity_count", "non_numeric_feature_count",
    "index_is_unique", "index_matches_target",
]
{
    "feature_dtype_counts": X_raw.dtypes.astype(str).value_counts().to_dict(),
    **{field: summary[field] for field in quality_fields},
}

## Target distribution (tabular)

In [ ]:
target_distribution = y.value_counts(dropna=False).rename("count").to_frame()
target_distribution["proportion"] = y.value_counts(
    dropna=False, normalize=True
)
target_distribution

## Per-feature audit

A feature is quasi-constant when one non-missing value accounts for at least 99% of non-missing observations and the feature is not strictly constant. Only a preview is displayed; the full table is written as an audit report.

In [ ]:
feature_audit = audit_numeric_features(X_raw, quasi_constant_threshold=0.99)
feature_audit_path = project_root / "reports/tables/feature_audit.csv"
feature_audit_path.parent.mkdir(parents=True, exist_ok=True)
feature_audit.to_csv(feature_audit_path, index=False)
feature_audit.head(10)

## Explicit float64 / float32 comparison

Relative error is computed only for finite, non-zero original values. Original zeros are excluded from that ratio to avoid artificial infinity or NaN, but exact changes at zero remain part of the changed-value count.

In [ ]:
X_float32 = convert_numeric_dtype(X_raw, dtype="float32", copy=True)
validate_dtype_conversion(X_raw, X_float32, target_dtype="float32")
dtype_comparison = compare_numeric_precision(X_raw, X_float32)
dtype_comparison

## Computed interpretation and provisional decision

The following text is generated from the computed validation flags and metrics. Float32 representation differences are reported for scientific review rather than automatically classified as data errors.

In [ ]:
safe_structure = all(
    dtype_comparison[key]
    for key in [
        "shape_preserved", "index_preserved", "columns_preserved",
        "missing_values_preserved", "infinities_preserved",
    ]
)
decision = (
    "Provisionally accept float32 for downstream feature storage, while retaining "
    "raw float64 loading and validating model sensitivity later."
    if safe_structure and dtype_comparison["memory_saved_mb"] > 0
    else "Retain float64 until structural preservation or memory benefit is resolved."
)
{
    "observed_memory_reduction_percentage": dtype_comparison["memory_reduction_percentage"],
    "observed_maximum_absolute_error": dtype_comparison["maximum_absolute_error"],
    "observed_maximum_relative_error": dtype_comparison["maximum_relative_error"],
    "decision": decision,
}

## Shared Train/Test Split

The common 80/20 split uses `random_state=42`, shuffling, and target stratification so both partitions preserve the observed class imbalance. No learned preprocessing occurs before this split, and the original row indices are retained for reproducibility.

**The test set is reserved for final evaluation and must not be used for model selection.**

In [ ]:
X_train, X_test, y_train, y_test = create_train_test_split(X_float32, y)
split_summary = validate_train_test_split(
    X_float32, y, X_train, X_test, y_train, y_test
)
{
    "parameters": {
        "test_size": split_summary["split_test_size"],
        "random_state": split_summary["split_random_state"],
        "stratified": split_summary["stratified"],
    },
    "sizes": {"train": split_summary["n_train"], "test": split_summary["n_test"]},
    "target_distributions": {
        "original": split_summary["original_target_distribution"],
        "train": split_summary["train_target_distribution"],
        "test": split_summary["test_target_distribution"],
    },
    "train_indices_sha256": split_summary["train_indices_sha256"],
    "test_indices_sha256": split_summary["test_indices_sha256"],
}